# SparkSession

SparkSession is automatically created when you start up a Notebook (e.g. Zeppelin, Databricks)

<img src="https://i.imgur.com/5Ai45fb.jpg" width=500px>


In [ ]:
%spark
# Scala SparkSession
spark

In [ ]:
%spark.pyspark
#PySpark SparkSession
spark

# Show DataFrame

`df.show()` is the Spark native API that displays data but it's not pretty.

`z.show(df)` is a Zeppelin build-in feature that allows you to show a df result in a pretty way


In [ ]:
%spark.pyspark

#List all hive tables in a df
tables_df = spark.sql("show tables")

In [ ]:
%spark.pyspark
z.show(tables_df)

# Spark SQL vs Dataframe

`%sql` is the Spark SQL interpreter

`%spark.pyspark` is the PySpark interpreter

`%spark` is the Spark Scala interpreter


In [ ]:
%sql

select count(1) as RecordCount from wdi_csv_parquet

In [ ]:
%spark.pyspark

#Read Hive data to a df (this is lazy)
wdi_df = spark.sql("SELECT * from wdi_csv_parquet")
#Persist df in memory for fast futuer access
wdi_df = wdi_df.cache()
wdi_df.printSchema()

#Spark action is eager
z.show(wdi_df.count())

# Show Historical GDP for Canada


In [ ]:
%sql
SELECT year, IndicatorValue as GDP
FROM wdi_csv_parquet
WHERE indicatorCode = 'NY.GDP.MKTP.KD.ZG' and countryName = 'Canada'
ORDER BY year

In [ ]:
%spark.pyspark 
# Historical GDP for Canada - PySpark
wdi_canada_df = wdi_df.filter((wdi_df.indicatorcode == 'NY.GDP.MKTP.KD.ZG') & (wdi_df.countryname == 'Canada')).select(wdi_df.year, (wdi_df.indicatorvalue.alias('GDP'))).orderBy(wdi_df.year)

# GDP over time - Bar Chart 
z.show(wdi_canada_df.select("year", "GDP"))

# Show GDP for Each County and Sort By Year


In [ ]:
%sql
SELECT countryname,
       year,
       indicatorcode,
       indicatorvalue
FROM wdi_csv_parquet
WHERE indicatorcode = 'NY.GDP.MKTP.KD.ZG'
DISTRIBUTE BY countryname
SORT BY countryname, year

In [ ]:
%spark.pyspark

# GDP for Each County and Sort By Year
wdi_gdp_all = wdi_df.filter((wdi_df.indicatorcode == 'NY.GDP.MKTP.KD.ZG')).select(wdi_df.countryname, wdi_df.year, wdi_df.indicatorvalue).orderBy(wdi_df.countryname,wdi_df.year)

z.show(wdi_gdp_all)

# Find the highest GDP for each country


In [ ]:
%sql

SELECT wdi_csv_parquet.indicatorvalue AS value, 
       wdi_csv_parquet.year           AS year, 
       wdi_csv_parquet.countryname    AS country 
FROM   (SELECT Max(indicatorvalue) AS ind, 
               countryname 
        FROM   wdi_csv_parquet 
        WHERE  indicatorcode = 'NY.GDP.MKTP.KD.ZG' 
               AND indicatorvalue <> 0 
        GROUP  BY countryname) t1 
       INNER JOIN wdi_csv_parquet 
               ON t1.ind = wdi_csv_parquet.indicatorvalue 
                  AND t1.countryname = wdi_csv_parquet.countryname

In [ ]:
%spark.pyspark

from pyspark.sql import functions as F

filtered_df = wdi_df.filter((wdi_df.indicatorcode == "NY.GDP.MKTP.KD.ZG") & (wdi_df.indicatorvalue != 0)) 

max_gdp_df = filtered_df.groupBy("countryname").agg(F.max("indicatorvalue").alias("ind"))

f = filtered_df.alias("f") 

m = max_gdp_df.alias("m") 

result_df = f.join(m, (F.col("f.countryname") == F.col("m.countryname")) & (F.col("f.indicatorvalue") == F.col("m.ind")))

final_df = result_df.select(F.col("f.countryname").alias("country"), F.col("f.year").alias("year"), F.col("f.indicatorvalue").alias("value")).orderBy("country") 

z.show(final_df)